# QuantPulse - Quickstart Notebook

This notebook walks through:
1. Triggering a historical backfill (5 years of OHLCV + macro)
2. Running the feature pipeline manually
3. Training the regime models
4. Inspecting regime signals

**Prerequisites:** `make up` has been run and all services are healthy.

In [ ]:
import httpx
import polars as pl
import json

API = 'http://localhost:8000'
INGESTION = 'http://localhost:8001'
FEATURE   = 'http://localhost:8002'
REGIME    = 'http://localhost:8003'

# Get auth token
resp = httpx.post(f'{API}/auth/token', data={'username': 'demo', 'password': 'demo'})
TOKEN = resp.json()['access_token']
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
print('Authenticated:', TOKEN[:20], '...')

In [ ]:
# Step 1 - Trigger 5-year historical backfill
resp = httpx.post(f'{INGESTION}/trigger/backfill?years=5', timeout=600)
print('Backfill triggered:', resp.json())
print('This will run in the background - check logs with: make logs-ingestion')

In [ ]:
# Step 2 - Trigger feature computation
resp = httpx.post(f'{FEATURE}/trigger/compute', timeout=300)
print('Feature pipeline:', resp.json())

In [ ]:
# Step 3 - Train regime models
resp = httpx.post(f'{REGIME}/train', timeout=10)
print('Training triggered:', resp.json())
print('Monitor in MLflow UI: http://localhost:5000')

In [ ]:
# Step 4 - Check regime signals
resp = httpx.get(f'{API}/api/v1/regime', headers=HEADERS)
regimes = resp.json()['regimes']

rows = []
for ticker, sig in regimes.items():
    rows.append({
        'ticker': ticker,
        'regime': sig['regime_name'],
        'confidence': round(sig['confidence'], 3),
        'updated_at': sig['updated_at'],
    })

df = pl.DataFrame(rows).sort('regime')
print(df)

In [ ]:
# Step 5 - Fetch OHLCV for SPY and inspect
resp = httpx.get(f'{API}/api/v1/ohlcv/SPY?limit=50', headers=HEADERS)
bars = resp.json()['bars']
df_spy = pl.DataFrame(bars)
print(df_spy.tail(10))

In [ ]:
# Step 6 - Check alerts
resp = httpx.get(f'{API}/api/v1/alerts?limit=20', headers=HEADERS)
alerts = resp.json()['alerts']
for a in alerts[:5]:
    print(f"[{a['severity']}] {a['alert_type']} | {a['ticker']} | {a['time']}")